# Notebook name

# Setup

## Imports

In [ ]:
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from thesis_project import (
    config,
)

## Paths

In [ ]:
TICKER = "fbtp"
# NOTE: Use "notebook-name" if needed
NOTEBOOK = None

# Lobs directories
LOB_DIR = config.RAW_DIR / TICKER
LOB_FUT_DIR = LOB_DIR / "futures"
LOB_CTD_DIR = LOB_DIR / "ctd"

# Output directories
if NOTEBOOK is not None:
    OUT_FIG_DIR = config.FIG_DIR / TICKER / NOTEBOOK
    OUT_FIG_DIR.mkdir(parents=True, exist_ok=True)
    OUT_TAB_DIR = config.TAB_DIR / TICKER / NOTEBOOK
    OUT_TAB_DIR.mkdir(parents=True, exist_ok=True)
    # NOTE: Create a purpose specific directory if needed
    # OUT_INT_DIR = config.INT_DIR / TICKER / NOTEBOOK
    # OUT_INT_DIR.mkdir(parents=True, exist_ok=True)
    # OUT_PRO_DIR = config.PRO_DIR / TICKER / NOTEBOOK
    # OUT_PRO_DIR.mkdir(parents=True, exist_ok=True)

## Settings

In [ ]:
# Matplotlib global settings
# config.MPL contains a personalized default configuration for matplotlib
plt.rcdefaults()
plt.style.use("seaborn-v0_8-paper")

In [ ]:
# Warnings settings
warnings.simplefilter("always", UserWarning)

## Utilities

### Plotting

# Main

## Common

In [ ]:
# Common data
cf_df = pd.read_csv(
    LOB_DIR / "daily_cf.csv", parse_dates=["Date"], date_format="%Y-%m-%d", index_col="Date"
)

ctd_meta_df = pd.read_csv(
    LOB_DIR / "ctd_metadata.csv", parse_dates=["Issue Date", "Maturity Date", "First Coupon Date"]
)

fut_meta_df = pd.read_csv(
    LOB_DIR / "fut_metadata.csv", parse_dates=["Last Trading Date", "Delivery Date"]
)

ctd_switch_df = pd.read_csv(LOB_DIR / "ctd_switch.csv", parse_dates=["CTD Switch Date"])

if TICKER == "fbtp":
    START_YEAR = config.FBTP_START_YEAR
    END_YEAR = config.FBTP_END_YEAR

if TICKER == "fbts":
    START_YEAR = config.FBTS_START_YEAR
    END_YEAR = config.FBTS_END_YEAR

## Load Targets

In [ ]:
microstructure_path = config.INT_DIR / TICKER / "data-microstructure"

dir = "basis"
filename = "gross_basis_freq_1s_from_2022_08_01_to_2024_12_30.parquet"
basis_df = pd.read_parquet(microstructure_path / dir / filename)
basis_df.rename(columns={"Basis": "basis"}, inplace=True)

dir = "irr"
filename = "irr_freq_1s_from_2022_08_01_to_2024_12_30.parquet"
irr_df = pd.read_parquet(microstructure_path / dir / filename)
irr_df.rename(columns={"IRR": "irr"}, inplace=True)

target_df = pd.merge(
    basis_df,
    irr_df,
    left_index=True,
    right_index=True,
    how="outer",
)

## Load Features

In [ ]:
LOB_TYPES = ["fut", "ctd"]

dir = "mid-price"
filename = "mid_price_freq_1s_from_2022_08_01_to_2024_12_30.parquet"
fut_mid_price_df = pd.read_parquet(microstructure_path / dir / f"fut_{filename}")
fut_mid_price_df.rename(columns={"Mid Price": "fut_mid_price"}, inplace=True)
ctd_mid_price_df = pd.read_parquet(microstructure_path / dir / f"ctd_{filename}")
ctd_mid_price_df.rename(columns={"Mid Price": "ctd_mid_price"}, inplace=True)
mid_price_df = pd.merge(
    fut_mid_price_df,
    ctd_mid_price_df,
    how="outer",
    left_index=True,
    right_index=True,
)

features_df = mid_price_df.copy()

dir = "micro-price"
filename = "micro_price_freq_1s_from_2022_08_01_to_2024_12_30.parquet"
fut_micro_price_df = pd.read_parquet(microstructure_path / dir / f"fut_{filename}")
fut_micro_price_df.rename(columns={"Micro Price": "fut_micro_price"}, inplace=True)
ctd_micro_price_df = pd.read_parquet(microstructure_path / dir / f"ctd_{filename}")
ctd_micro_price_df.rename(columns={"Micro Price": "ctd_micro_price"}, inplace=True)
micro_price_df = pd.merge(
    fut_micro_price_df,
    ctd_micro_price_df,
    how="outer",
    left_index=True,
    right_index=True,
)

features_df = features_df.merge(
    micro_price_df,
    how="outer",
    left_index=True,
    right_index=True,
)

dir = "spread"
filename = "spread_freq_1s_from_2022_08_01_to_2024_12_30.parquet"
fut_spread_df = pd.read_parquet(microstructure_path / dir / f"fut_{filename}")
fut_spread_df.rename(columns={"Spread": "fut_spread"}, inplace=True)
ctd_spread_df = pd.read_parquet(microstructure_path / dir / f"ctd_{filename}")
ctd_spread_df.rename(columns={"Spread": "ctd_spread"}, inplace=True)
spread_df = pd.merge(
    fut_spread_df,
    ctd_spread_df,
    how="outer",
    left_index=True,
    right_index=True,
)

features_df = features_df.merge(
    spread_df,
    how="outer",
    left_index=True,
    right_index=True,
)

dir = "obi"
filename = "obi_lvl_1_scaled_freq_1s_from_2022_08_01_to_2024_12_30.parquet"
fut_obi_df = pd.read_parquet(microstructure_path / dir / f"fut_{filename}")
fut_obi_df.rename(columns={"OBI": "fut_obi"}, inplace=True)
ctd_obi_df = pd.read_parquet(microstructure_path / dir / f"ctd_{filename}")
ctd_obi_df.rename(columns={"OBI": "ctd_obi"}, inplace=True)
obi_df = pd.merge(
    fut_obi_df,
    ctd_obi_df,
    how="outer",
    left_index=True,
    right_index=True,
)

features_df = features_df.merge(
    obi_df,
    how="outer",
    left_index=True,
    right_index=True,
)

dir = "ofi"
filename = "ofi_lvl_1_freq_1s_from_2022_08_01_to_2024_12_30.parquet"
fut_ofi_df = pd.read_parquet(microstructure_path / dir / f"fut_{filename}")
cols = fut_ofi_df.columns
new_cols_map = {col: "fut_" + col.lower() for col in cols}
fut_ofi_df.rename(columns=new_cols_map, inplace=True)
ctd_ofi_df = pd.read_parquet(microstructure_path / dir / f"ctd_{filename}")
new_cols_map = {col: "ctd_" + col.lower() for col in cols}
ctd_ofi_df.rename(columns=new_cols_map, inplace=True)
ofi_df = pd.merge(
    fut_ofi_df,
    ctd_ofi_df,
    how="outer",
    left_index=True,
    right_index=True,
)

features_df = features_df.merge(
    ofi_df,
    how="outer",
    left_index=True,
    right_index=True,
)

## Filter Dates

In [ ]:
# import datetime

# # Cut a certain window around rollover days
# ROLL_DAYS_BEFORE = 5
# ROLL_DAYS_AFTER = 0


# def build_exclusion_dates(
#     config_dates: dict, fut_metadata: pd.DataFrame, ticker: str
# ) -> list[datetime.date]:
#     # Load dates with structural isses (from src/thesis_project/config.py)
#     structural_dates = pd.to_datetime(config_dates.get(ticker, [])).date.tolist()

#     # Load last trading days for futures contract and build a window around them to exclude
#     rollover_dates = set()
#     last_trading_dates = pd.to_datetime(fut_metadata["Last Trading Date"]).dropna().dt.date

#     # Consider only dates in the specified range
#     mask = (last_trading_dates >= pd.to_datetime(START_DATE).date()) & (
#         last_trading_dates <= pd.to_datetime(END_DATE).date()
#     )
#     last_trading_dates = last_trading_dates[mask]

#     for d in last_trading_dates:
#         start_excl = d - datetime.timedelta(days=ROLL_DAYS_BEFORE)
#         end_excl = d + datetime.timedelta(days=ROLL_DAYS_AFTER)

#         # Build the exact calendar day range
#         delta = (end_excl - start_excl).days
#         date_range = [start_excl + datetime.timedelta(days=x) for x in range(delta + 1)]
#         rollover_dates.update(date_range)

#     unified_exclusions = list(set(structural_dates).union(rollover_dates))

#     print(f"Total structural problematic excluded dates: {len(structural_dates)}")
#     print(f"Total last trading neighbor excluded dates: {len(rollover_dates)}")
#     print(f"Combined unique excluded dates: {len(unified_exclusions)}")

#     return unified_exclusions


# # Compute dates to drop
# excluded_dates_list = build_exclusion_dates(config.DATES_TO_EXCLUDE, fut_meta_df, TICKER)

In [ ]:
# # Apply Date Filtering early to avoid computing features over junk/rollover regions
# exclusions_series = pl.Series("excluded_date", excluded_dates_list)
# lf_filtered = lf_joined.filter(~pl.col("date").is_in(exclusions_series))
# print(f"Exclusion filter applied: {len(excluded_dates_list)} dates excluded")

## Preprocessing

In [ ]:
# For testing purposes, we cut the dataset
start_date = "2022-09-12"
end_date = "2022-09-12"

target_df = target_df.loc[start_date:end_date]
features_df = features_df.loc[start_date:end_date]

In [ ]:
TARGETS_TIME_DELTAS = [
    # 10, # seconds
    # 30,
    60,
]

for delta in TARGETS_TIME_DELTAS:
    col = f"basis_delta_{delta}s"
    target_df[col] = target_df["basis"].groupby(pd.Grouper(freq="D")).diff(delta)

    col = f"irr_delta_{delta}s"
    target_df[col] = target_df["irr"].groupby(pd.Grouper(freq="D")).diff(delta)
target_df = target_df.drop(columns=["basis", "irr"])

In [ ]:
from tqdm.notebook import tqdm

FEATURES_TIME_DELTAS = [
    # 10, # seconds
    # 30,
    60,
]

ROLLING_STATS_WINDOWS = [
    60,  # seconds
    # 300,
    # 600,
]

ROLLING_WINDOWS_STATS = [
    "mean",
    "std",
    "min",
    "max",
]

START_TIME = "09:00"
END_TIME = "17:00"

offset = round(max(ROLLING_STATS_WINDOWS) * 1.5)
features_df = features_df.between_time(
    (pd.to_datetime(START_TIME) - pd.Timedelta(seconds=offset)).time(),
    pd.to_datetime(END_TIME).time(),
)

new_features = {}
for feature in tqdm(features_df.columns, desc="Processing features", unit="feature"):
    grouped = features_df[feature].groupby(pd.Grouper(freq="D"))

    # First differences
    for delta in tqdm(
        TARGETS_TIME_DELTAS, leave=False, desc="Processing time deltas", unit="delta"
    ):
        col = f"{feature}_delta_{delta}s"
        new_features[col] = grouped.diff(delta)

    # Rolling statistics
    for window in tqdm(
        ROLLING_STATS_WINDOWS, leave=False, desc="Processing rolling stats", unit="window"
    ):
        min_periods = window // 2

        rolled = grouped.rolling(window, min_periods=min_periods)

        rolling_stats = rolled.agg(ROLLING_WINDOWS_STATS).reset_index(level=0, drop=True)

        for stat in ROLLING_WINDOWS_STATS:
            new_features[f"{feature}_rolling_{stat}_{window}s"] = rolling_stats[stat]

features_df = pd.concat([features_df, pd.DataFrame(new_features)], axis=1)

## Prediction Analysis

In [ ]:
# Filter data
target_df = target_df.between_time(START_TIME, END_TIME)
features_df = features_df.between_time(START_TIME, END_TIME)

### Serial

In [ ]:
import infomeasure as im

PREDICTION_TIME_HORIZIONS = [
    10,
    30,
    60,
]

target_df = target_df.between_time(START_TIME, END_TIME)
features_df = features_df.between_time(START_TIME, END_TIME)

results = {}
for h in tqdm(PREDICTION_TIME_HORIZIONS, desc="Processing prediction horizons", unit="horizon"):
    for y_col in tqdm(
        target_df.columns, desc="Processing target columns", leave=False, unit="target"
    ):
        z = target_df[y_col]
        y = z.shift(-h)

        for x_col in tqdm(
            features_df.columns, desc="Processing feature columns", leave=False, unit="feature"
        ):
            x = features_df[x_col]

            aligned = pd.concat(
                [x, y, z],
                axis=1,
                join="outer",
            ).dropna()
            aligned.columns = ["x", "y", "z"]

            x = aligned["x"]
            y = aligned["y"]
            z = aligned["z"]

            # Compute Pearson correlation
            pearson_corr = y.corr(x)

            # Mutual Information
            mi = im.mutual_information(
                x,
                y,
                approach="ksg",
            )

            # Conditional Mutual Information
            cmi = im.conditional_mutual_information(
                x,
                y,
                cond=z,
                approach="ksg",
            )

            # Transfer Entropy
            # te = im.transfer_entropy(
            #     x,
            #     y,
            #     approach="kernel",
            # )

            # Conditional Transfer Entropy
            # cte = im.conditional_transfer_entropy(
            #     x,
            #     y,
            #     cond=z,
            #     approach="kernel",
            # )

            print(f"Horizon: {h} seconds \nTarget: {y_col} \nFeature: {x_col}")
            print(f"    Pearson Correlation: {pearson_corr:.4f}")
            print(f"    Mutual Information: {mi:.4f}\n")
            print(f"    Conditional Mutual Information: {cmi:.4f}\n")
            # print(f"    Transfer Entropy: {te:.4f}\n")
            # print(f"    Conditional Transfer Entropy: {cte:.4f}\n")

            results[(h, y_col, x_col)] = {
                "pearson_correlation": pearson_corr,
                "mutual_information": mi,
                "conditional_mutual_information": cmi,
                # "transfer_entropy": te,
                # "conditional_transfer_entropy": cte,
            }

results_df = pd.DataFrame.from_dict(results, orient="index").reset_index()
results_df.rename(
    columns={"level_0": "horizon", "level_1": "target", "level_2": "feature"}, inplace=True
)

### Parallel

In [ ]:
import infomeasure as im
from joblib import Parallel, delayed
from tqdm.auto import tqdm

PREDICTION_TIME_HORIZIONS = [
    30,
    # 60,
    # 300,
]


# COntiguous numpy arrays should be faster
features_np = {
    col: np.ascontiguousarray(features_df[col].to_numpy(dtype=np.float64))
    for col in features_df.columns
}

targets_np = {
    col: np.ascontiguousarray(target_df[col].to_numpy(dtype=np.float64))
    for col in target_df.columns
}


# Shifted targets for each horizon
shifted_targets_np = {}

for h in PREDICTION_TIME_HORIZIONS:
    shifted_targets_np[h] = {}

    for col in target_df.columns:
        shifted = target_df[col].shift(-h)
        shifted_targets_np[h][col] = np.ascontiguousarray(shifted.to_numpy(dtype=np.float64))

print("Total targets: ", len(target_df.columns))
print("Total features: ", len(features_df.columns))
print("Total prediction horizons: ", len(PREDICTION_TIME_HORIZIONS))
print(
    f"Total (feature,target) pairs: {len(features_df.columns) * len(target_df.columns) * len(PREDICTION_TIME_HORIZIONS)}"
)


# Main function to parallelize
def compute_metrics(
    h: int,
    y_col: str,
    x_col: str,
):
    try:
        z = targets_np[y_col]
        y = shifted_targets_np[h][y_col]
        x = features_np[x_col]

        # Filter for valid samples only
        mask = np.isfinite(x) & np.isfinite(y) & np.isfinite(z)

        if not mask.any():
            warnings.warn(
                f"No valid samples for horizon {h}, target {y_col}, feature {x_col}", UserWarning
            )
            return None

        x_valid = x[mask]
        y_valid = y[mask]
        z_valid = z[mask]

        # Correlation
        pearson_corr = np.corrcoef(
            x_valid,
            y_valid,
        )[0, 1]

        # Mutual Information
        mi = im.mutual_information(
            x_valid,
            y_valid,
            approach="kernel",
            bandwidth=0.2,
            kernel="box",
        )

        # Conditional Mutual Information
        cmi = im.conditional_mutual_information(
            x_valid,
            y_valid,
            cond=z_valid,
            approach="kernel",
            bandwidth=0.2,
            kernel="box",
        )

        # # Transfer Entropy
        # te = im.transfer_entropy(
        #     x_valid,
        #     y_valid,
        #     approach="kernel",
        # )

        # Conditional Transfer Entropy
        # cte = im.conditional_transfer_entropy(
        #     x_valid,
        #     y_valid,
        #     cond=z_valid,
        #     approach="kernel",
        # )

        return {
            "horizon": h,
            "target": y_col,
            "feature": x_col,
            "pearson_correlation": pearson_corr,
            "mutual_information": mi,
            "conditional_mutual_information": cmi,
            # "transfer_entropy": te,
            # "conditional_transfer_entropy": cte,
            "n_samples": len(x_valid),
        }

    except Exception as e:
        return {
            "horizon": h,
            "target": y_col,
            "feature": x_col,
            "pearson_correlation": np.nan,
            "mutual_information": np.nan,
            "conditional_mutual_information": np.nan,
            # "transfer_entropy": np.nan,
            # "conditional_transfer_entropy": np.nan,
            "n_samples": 0,
            "error": str(e),
        }


# Parallelize
tasks = [
    (h, y_col, x_col)
    for h in PREDICTION_TIME_HORIZIONS
    for y_col in target_df.columns
    for x_col in features_df.columns
]

results = Parallel(
    n_jobs=-1,
    backend="loky",
    verbose=10,
)(
    delayed(compute_metrics)(
        h,
        y_col,
        x_col,
    )
    for h, y_col, x_col in tqdm(
        tasks,
        desc="Processing tasks",
        unit="task",
    )
)

# Drop out empty results
results_filtered = [r for r in results if r is not None]

results_df = pd.DataFrame(results_filtered)

# 'base' microstructure features (14)
# target (2)
# time horizons (3)
# time deltas for features and target (3,3)
# rolling stats (4)
# rolling stats windows (3)
# Metrics (3+2)

In [ ]:
results_df.sort_values(
    by=[
        "horizon",
        "target",
        "feature",
    ],
    inplace=True,
)
# results_df.set_index(
#     ["horizon", "target", "feature"],
#     inplace=True,
# )

results_df.to_parquet(config.INT_DIR / TICKER / "feature-engineering" / "results.parquet")

In [ ]:
results_df = pd.read_parquet(config.INT_DIR / TICKER / "feature-engineering" / "results.parquet")
# results_df.set_index(
#     ["horizon", "target", "feature"],
#     inplace=True,
# )
# display(results_df.head(10).sort_values(by="mutual_information", ascending=False))
results_df.sort_values(by="mutual_information", ascending=False, inplace=True)
results_df.head(10)